# 13장 최적화 — 경사하강과 학습률 — 수리 도감 코랩 실습

원자료: 연구소 수학 노트 MATH-13 「Optimization」 · 해설 페이지: https://adminmrobo.github.io/math_basic_for_ai_dictionary/13_optimization.html

위에서부터 셀을 차례로 실행하세요(Shift + Enter). 그래프는 영어 글자로 그렸습니다(코랩에 한글 글꼴을 따로 깔지 않아도 되게).

글 · 안상선 ((주)M-Robo 대표)


## 1. 학습률에 따른 경사하강 (f = θ^2)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def gd_path(theta, alpha, steps):
    path = [theta]
    for _ in range(steps):
        theta = theta - alpha * 2 * theta       # ∇f = 2θ
        path.append(theta)
    return np.array(path)

for a in [0.01, 0.1, 0.7, 1.0, 1.1]:
    p = gd_path(4.0, a, 20); r = 1 - 2 * a
    kind = "발산" if abs(r) > 1 else "제자리 왕복" if abs(r) == 1 else ("지그재그 수렴" if r < 0 else "수렴")
    print(f"α={a:<5} 공비 {r:+.2f} -> 20걸음 뒤 θ = {p[-1]:+.4g} ({kind})")
    plt.plot(p ** 2 + 1e-12, label=f"alpha={a}")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Learning rate matters"); plt.show()

## 2. 선형 회귀를 경사하강으로 (5장과 같은 데이터)


In [ ]:
x = np.array([1, 2, 3.0]); y = np.array([2, 3, 5.0])
def grads(w, b):
    err = y - (w * x + b)
    return -2 * np.mean(x * err), -2 * np.mean(err), np.mean(err ** 2)
w = b = 0.0
for t in range(5000):
    gw, gb, J = grads(w, b)
    w, b = w - 0.05 * gw, b - 0.05 * gb
    if t in (0, 10, 100, 1000, 4999):
        print(f"걸음 {t:>4}: w = {w:.4f}, b = {b:.4f}, MSE = {J:.5f}")
X = np.column_stack([np.ones_like(x), x])
print("정규방정식 답:", np.linalg.solve(X.T @ X, X.T @ y).round(4), "(b, w)")

## 3. 배치 vs SGD vs 미니배치


In [ ]:
rng = np.random.default_rng(0)
xd = rng.uniform(0, 2, 1000); yd = 3 * xd + 2 + rng.normal(0, 0.5, 1000)
def train(batch, epochs=20, lr=0.05):
    w = b = 0.0; hist = []
    for _ in range(epochs):
        idx = rng.permutation(len(xd))
        for s in range(0, len(xd), batch):
            j = idx[s:s + batch]; err = yd[j] - (w * xd[j] + b)
            w += lr * 2 * np.mean(xd[j] * err); b += lr * 2 * np.mean(err)
        hist.append(np.mean((yd - (w * xd + b)) ** 2))
    return w, b, hist
for name, bs in [("batch", 1000), ("SGD", 1), ("mini-batch 32", 32)]:
    w_, b_, h = train(bs)
    print(f"{name:14s}: w = {w_:.3f}, b = {b_:.3f}, 마지막 MSE = {h[-1]:.4f}")
    plt.plot(h, label=name)
plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend(); plt.title("Batch vs SGD vs mini-batch"); plt.show()

## 4. 좁고 긴 그릇에서 GD vs 모멘텀 vs Adam


In [ ]:
f = lambda p: p[0] ** 2 + 10 * p[1] ** 2
g = lambda p: np.array([2 * p[0], 20 * p[1]])
def run(method, lr, steps=300):
    p = np.array([5.0, 1.0]); v = np.zeros(2); m = np.zeros(2); s = np.zeros(2)
    for t in range(1, steps + 1):
        gr = g(p)
        if method == "GD":
            p = p - lr * gr
        elif method == "Momentum":
            v = 0.9 * v + gr; p = p - lr * v
        else:  # Adam
            m = 0.9 * m + 0.1 * gr; s = 0.999 * s + 0.001 * gr ** 2
            mh, sh = m / (1 - 0.9 ** t), s / (1 - 0.999 ** t)
            p = p - lr * mh / (np.sqrt(sh) + 1e-8)
        if f(p) < 1e-6:
            return t
    return steps
for meth, lr in [("GD", 0.01), ("Momentum", 0.01), ("Adam", 0.1)]:
    print(f"{meth:8s} (lr={lr}): 손실 < 0.000001 까지 {run(meth, lr, 3000)}걸음")
print("GD를 lr=0.09로 잘 맞추면:", run("GD", 0.09), "걸음 -> 학습률을 잘 고르면 기본 GD도 빠르다.")
print("개선판의 장점: 학습률을 덜 까다롭게 골라도 잘 움직인다(같은 lr=0.01에서 모멘텀이 약 3배 빠름).")

## 생성 AI 실습 프롬프트

아래 글을 코랩의 Gemini 창(또는 다른 생성 AI)에 붙여 넣고, 만들어진 코드를 위 코드의 결과와 비교해 보세요.

```text
너는 고등학생에게 경사하강법을 가르치는 파이썬 선생님이야. 구글 코랩에서 실행되는 코드를 만들어 줘.
주제: 경사하강, 학습률, SGD/미니배치, 선형 회귀 최적화
요구 사항:
1. f(θ) = θ²에 대해 θ₀ = 4에서 학습률 0.01, 0.1, 0.7, 1.0, 1.1로 각각 20걸음 경사하강하고, 걸음별 손실 곡선을 한 그림에 그려 줘(로그 축, 영어 범례). 각 경우가 수렴/지그재그/왕복/발산 중 무엇인지 출력해 줘.
2. 점 (1,2), (2,3), (3,5)에 대해 MSE 손실 J(w, b)의 그래디언트를 직접 계산하는 함수를 만들고, w=b=0, 학습률 0.05에서 5000걸음 경사하강해 w≈1.5, b≈0.333에 다가가는지 정규방정식 답과 비교해 줘.
3. 가상 데이터 1000개(y = 3x + 2 + 잡음, seed 고정)에 대해 배치·SGD·미니배치(32) 경사하강을 각각 같은 에포크 수만큼 돌려 손실 곡선을 비교해 줘.
4. numpy로 모멘텀과 Adam 갱신 규칙을 직접 구현해서, 좁고 긴 그릇 f(x, y) = x² + 10y²에서 기본 경사하강과 걸음 수를 비교해 줘.
5. 모든 줄에 한국어 주석.
```
